# 03. 回収率（ROI）分析
モデル予測確率・各種条件での単勝回収率を評価し、プラス期待値の買い目を探索する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys; sys.path.append('../src')
from evaluate_roi import roi_by_bet_condition, roi_from_model_proba, kelly_bet_sizes, roi_by_factor

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features_with_pred.csv', encoding='utf-8-sig')
# OOFが計算されている行のみ使用（最初のfoldはskip）
df = df[df['pred_win_proba'] > 0].copy()
print(f'shape: {df.shape}, races: {df["race_id"].nunique()}')

## 1. 予測確率閾値 vs 回収率

In [ ]:
thresholds = np.arange(0.05, 0.60, 0.025)
results = []
for thr in thresholds:
    r = roi_from_model_proba(df, df['pred_win_proba'].values, threshold=thr, bet_col='payout_win')
    r['threshold'] = thr
    results.append(r)
thr_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
thr_df.plot(x='threshold', y='roi', ax=axes[0], title='Threshold vs ROI (%)', marker='o')
axes[0].axhline(100, color='red', linestyle='--', label='break-even')
axes[0].legend()
thr_df.plot(x='threshold', y='n_bets', ax=axes[1], title='Threshold vs # Bets', marker='o', color='orange')
thr_df.plot(x='threshold', y='hit_rate', ax=axes[2], title='Threshold vs Hit Rate (%)', marker='o', color='green')
plt.tight_layout()
plt.show()
print(thr_df[['threshold','n_bets','hit_rate','roi']].to_string(index=False))

## 2. 予測確率 × オッズ帯域 回収率マトリクス

In [ ]:
df['pred_bin'] = pd.cut(df['pred_win_proba'],
                         bins=[0, 0.08, 0.15, 0.25, 0.40, 1.0],
                         labels=['<8%', '8-15%', '15-25%', '25-40%', '>40%'])
df['odds_bin'] = pd.cut(df['単勝オッズ'],
                         bins=[1, 3, 6, 10, 20, 999],
                         labels=['1-3x', '3-6x', '6-10x', '10-20x', '20x+'])

matrix_roi = df.groupby(['pred_bin', 'odds_bin'], observed=True).apply(
    lambda g: roi_by_bet_condition(g, pd.Series(True, index=g.index), bet_col='payout_win')['roi']
).unstack()

matrix_n = df.groupby(['pred_bin', 'odds_bin'], observed=True).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(matrix_roi, annot=True, fmt='.0f', cmap='RdYlGn', center=100,
            linewidths=0.5, ax=axes[0], cbar_kws={'label': 'ROI (%)'})
axes[0].set_title('ROI Matrix: Predicted Prob x Odds Band')
axes[0].set_xlabel('Odds Band')
axes[0].set_ylabel('Predicted Win Probability')

sns.heatmap(matrix_n, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('# Horses: Predicted Prob x Odds Band')
axes[1].set_xlabel('Odds Band')

plt.tight_layout()
plt.show()

## 3. オッズ乖離（割安・割高）で絞った場合の回収率

In [ ]:
# log_odds_gap > 0 = 実際のオッズが予想より高い = 市場が過小評価（割安）
rows = []
for gap_min in np.arange(-1.0, 1.25, 0.25):
    mask = df['log_odds_gap'] >= gap_min
    r = roi_by_bet_condition(df, mask, bet_col='payout_win')
    r['log_odds_gap_min'] = round(gap_min, 2)
    rows.append(r)
gap_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gap_df.plot(x='log_odds_gap_min', y='roi', ax=axes[0], marker='o',
            title='ROI by Min Log Odds Gap (>=threshold)')
axes[0].axhline(100, color='red', linestyle='--')
axes[0].set_xlabel('log_odds_gap threshold')
gap_df.plot(x='log_odds_gap_min', y='n_bets', ax=axes[1], marker='o', color='orange',
            title='# Bets by Min Log Odds Gap')
plt.tight_layout()
plt.show()
print(gap_df[['log_odds_gap_min','n_bets','hit_rate','roi']].to_string(index=False))

## 4. ケリー基準によるベットサイズ分析

In [ ]:
kelly = kelly_bet_sizes(df['pred_win_proba'].values, df['単勝オッズ'].values, fraction=0.25)
df['kelly'] = kelly
kelly_mask = df['kelly'] > 0
print(f'ケリー正例数: {kelly_mask.sum()} / {len(df)} ({kelly_mask.mean()*100:.1f}%)')

r = roi_by_bet_condition(df, kelly_mask, bet_col='payout_win')
print('\n[正ケリー馬 フラットベット]')
for k, v in r.items(): print(f'  {k}: {v}')

## 5. 戦略別 累積損益シミュレーション

In [ ]:
STRATEGIES = {
    'All horses (baseline)':  pd.Series(True, index=df.index),
    '1st popularity only':    df['単勝人気'] == 1,
    'High score (top 30%)':   df['得点_rank_pct'] >= 0.7,
    'Model pred > 15%':       df['pred_win_proba'] > 0.15,
    'Model pred > 25%':       df['pred_win_proba'] > 0.25,
    'Positive Kelly':         kelly_mask,
    'Undervalued (gap>0.3)':  df['log_odds_gap'] > 0.3,
}

plt.figure(figsize=(14, 6))
for label, mask in STRATEGIES.items():
    sub = df[mask].sort_values('race_id').copy()
    sub['net'] = sub['payout_win'] - 100
    plt.plot(sub['net'].cumsum().values, label=label, alpha=0.85)

plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Cumulative P&L by Strategy (100 JPY/bet)')
plt.xlabel('# Bets')
plt.ylabel('Cumulative P&L (JPY)')
plt.legend()
plt.tight_layout()
plt.show()

print('\n[Strategy Summary]')
print(f'{"Strategy":<28} {"N Bets":>7} {"Hit%":>6} {"ROI%":>7}')
print('-' * 55)
for label, mask in STRATEGIES.items():
    r = roi_by_bet_condition(df, mask, bet_col='payout_win')
    print(f'{label:<28} {r["n_bets"]:>7} {r["hit_rate"]:>6.1f} {r["roi"]:>7.1f}')

## 6. 主要ファクター別 ROI（連続変数ビニング）

In [ ]:
factors = ['log_odds', '得点', '騎手評価', '予想タイム指数', '先行指数', '波乱度']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, col in zip(axes.flat, factors):
    try:
        roi_df = roi_by_factor(df, col, bins=8, bet_col='payout_win')
        roi_df['roi'].plot(kind='bar', ax=ax, title=f'{col} vs ROI (%)')
        ax.axhline(100, color='red', linestyle='--', linewidth=0.8)
        ax.set_xlabel(col)
        ax.tick_params(axis='x', rotation=45)
    except Exception as e:
        ax.set_title(f'{col}: error')

plt.tight_layout()
plt.show()

## 7. 複勝（3着以内）回収率分析

In [ ]:
# 複勝：3着以内に入れば払戻
thresholds2 = np.arange(0.10, 0.70, 0.05)
results2 = []
for thr in thresholds2:
    mask = df['pred_win_proba'] >= thr
    r = roi_by_bet_condition(df, mask, bet_col='payout_place')
    r['threshold'] = thr
    results2.append(r)
thr2_df = pd.DataFrame(results2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
thr2_df.plot(x='threshold', y='roi', ax=axes[0], marker='o', color='purple',
             title='Place Bet ROI by Model Threshold (%)')
axes[0].axhline(100, color='red', linestyle='--')
thr2_df.plot(x='threshold', y='hit_rate', ax=axes[1], marker='o', color='teal',
             title='Place Hit Rate by Model Threshold (%)')
plt.tight_layout()
plt.show()
print(thr2_df[['threshold','n_bets','hit_rate','roi']].to_string(index=False))

## 8. まとめ：回収率向上のための洞察

In [ ]:
print('=' * 60)
print('回収率向上ファクター分析 サマリー（2022年中央競馬）')
print('=' * 60)

summary = []
for label, mask in STRATEGIES.items():
    r = roi_by_bet_condition(df, mask, bet_col='payout_win')
    summary.append({'Strategy': label, 'N': r['n_bets'],
                    'Hit%': r['hit_rate'], 'ROI%': r['roi']})

summary_df = pd.DataFrame(summary).sort_values('ROI%', ascending=False)
print(summary_df.to_string(index=False))
print()
print('推奨ファクター優先順位:')
print('  1. 得点・デフォルト得点（最も着順相関が高い）')
print('  2. 予想タイム指数（LightGBMでも上位特徴量）')
print('  3. log_odds_gap（市場の過小評価を利用）')
print('  4. 騎手評価（人的要素で最重要）')
print('  5. 先行指数・予想展開（展開利不利）')
print('  6. 前走着順・前走レースレベル（近走の勢い）')